In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('../data/processed/loans_clean.csv')
conn = sqlite3.connect('../data/processed/loans.db')
df.to_sql('loans', conn, if_exists='replace', index=False)

print(pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM loans", conn))

   total_rows
0       38576


In [2]:
def q(sql):
    return pd.read_sql_query(sql, conn)

In [3]:
q("""
SELECT issue_month,
       COUNT(*) AS loans,
       SUM(loan_amount) AS total_funded,
       ROUND(AVG(loan_amount), 0) AS avg_loan
FROM loans
GROUP BY issue_month
ORDER BY issue_month;
""")

,issue_month,loans,total_funded,avg_loan
0,2021-01,2332,25031650,10734.0
1,2021-02,2279,24647825,10815.0
2,2021-03,2627,28875700,10992.0
3,2021-04,2755,29800800,10817.0
4,2021-05,2911,31738350,10903.0
5,2021-06,3184,34161475,10729.0
6,2021-07,3366,35813900,10640.0
7,2021-08,3441,38149600,11087.0
8,2021-09,3536,40907725,11569.0
9,2021-10,3796,44893800,11827.0


In [4]:
q("""
SELECT grade,
       COUNT(*) AS loans_with_outcome,
       SUM(default_flag) AS defaults,
       ROUND(100.0 * SUM(default_flag) / COUNT(*), 2) AS default_rate_pct
FROM loans
WHERE default_flag IS NOT NULL
GROUP BY grade
ORDER BY grade;
""")

,grade,loans_with_outcome,defaults,default_rate_pct
0,A,9654,552.0,5.72
1,B,11347,1343.0,11.84
2,C,7647,1266.0,16.56
3,D,4966,1072.0,21.59
4,E,2611,691.0,26.46
5,F,957,311.0,32.50
6,G,296,98.0,33.11


In [5]:
q("""
WITH monthly AS (
    SELECT issue_month, COUNT(*) AS loans
    FROM loans
    GROUP BY issue_month
)
SELECT issue_month,
       loans,
       LAG(loans) OVER (ORDER BY issue_month) AS prev_month_loans,
       ROUND(100.0 * (loans - LAG(loans) OVER (ORDER BY issue_month))
             / LAG(loans) OVER (ORDER BY issue_month), 1) AS growth_pct
FROM monthly;
""")

,issue_month,loans,prev_month_loans,growth_pct
0,2021-01,2332,NaN,NaN
1,2021-02,2279,2332.0,-2.3
2,2021-03,2627,2279.0,15.3
3,2021-04,2755,2627.0,4.9
4,2021-05,2911,2755.0,5.7
5,2021-06,3184,2911.0,9.4
6,2021-07,3366,3184.0,5.7
7,2021-08,3441,3366.0,2.2
8,2021-09,3536,3441.0,2.8
9,2021-10,3796,3536.0,7.4


In [6]:
q("""
SELECT region,
       COUNT(*) AS loans,
       SUM(loan_amount) AS total_funded,
       ROUND(AVG(loan_amount), 0) AS avg_loan,
       ROUND(100.0 * SUM(default_flag) / COUNT(default_flag), 2) AS default_rate_pct
FROM loans
GROUP BY region
ORDER BY total_funded DESC;
""")

,region,loans,total_funded,avg_loan,default_rate_pct
0,South,12651,143295125,11327.0,14.20
1,West,11067,124977525,11293.0,15.51
2,Northeast,9459,107361850,11350.0,13.11
3,Midwest,5399,60122575,11136.0,13.63


In [7]:
q("""
SELECT address_state,
       COUNT(*) AS loans,
       SUM(loan_amount) AS total_funded,
       ROUND(AVG(loan_amount), 0) AS avg_loan
FROM loans
GROUP BY address_state
ORDER BY total_funded DESC
LIMIT 5;
""")

,address_state,loans,total_funded,avg_loan
0,CA,6894,78484125,11384.0
1,NY,3701,42077050,11369.0
2,TX,2664,31236650,11725.0
3,FL,2773,30046125,10835.0
4,NJ,1822,21657475,11887.0


In [8]:
q("""
SELECT CASE WHEN emp_length_years = 0 THEN '0 (under 1 yr)'
            WHEN emp_length_years <= 4 THEN '1-4 yrs'
            WHEN emp_length_years <= 9 THEN '5-9 yrs'
            ELSE '10+ yrs' END AS emp_group,
       COUNT(default_flag) AS loans_with_outcome,
       ROUND(100.0 * SUM(default_flag) / COUNT(default_flag), 2) AS default_rate_pct
FROM loans
GROUP BY emp_group
ORDER BY MIN(emp_length_years);
""")

,emp_group,loans_with_outcome,default_rate_pct
0,0 (under 1 yr),4500,14.02
1,1-4 yrs,14782,13.58
2,5-9 yrs,9717,14.12
3,10+ yrs,8479,15.59


In [9]:
q("""
SELECT verification_status,
       COUNT(default_flag) AS loans_with_outcome,
       ROUND(100.0 * SUM(default_flag) / COUNT(default_flag), 2) AS default_rate_pct
FROM loans
GROUP BY verification_status
ORDER BY default_rate_pct DESC;
""")

,verification_status,loans_with_outcome,default_rate_pct
0,Verified,11754,16.47
1,Source Verified,9480,14.58
2,Not Verified,16244,12.40


In [10]:
q("""
SELECT term_months,
       COUNT(*) AS loans,
       ROUND(AVG(int_rate) * 100, 2) AS avg_interest_pct,
       ROUND(100.0 * SUM(default_flag) / COUNT(default_flag), 2) AS default_rate_pct
FROM loans
GROUP BY term_months;
""")

,term_months,loans,avg_interest_pct,default_rate_pct
0,36,28237,11.03,10.71
1,60,10339,14.83,25.00


In [11]:
q("""
SELECT loan_status,
       COUNT(*) AS loans
FROM loans
WHERE dti_pct >= 20
GROUP BY loan_status;
""")

,loan_status,loans
0,Charged Off,1137
1,Current,283
2,Fully Paid,5871


In [12]:
q("""
SELECT grade,
       purpose,
       COUNT(default_flag) AS loans_with_outcome,
       ROUND(100.0 * SUM(default_flag) / COUNT(default_flag), 2) AS default_rate_pct
FROM loans
WHERE default_flag IS NOT NULL
GROUP BY grade, purpose
HAVING COUNT(default_flag) >= 100
ORDER BY default_rate_pct DESC
LIMIT 5;
""")

,grade,purpose,loans_with_outcome,default_rate_pct
0,F,Debt consolidation,547,35.47
1,E,small business,184,34.78
2,G,Debt consolidation,162,33.95
3,D,small business,277,32.49
4,E,other,204,30.88
